In [84]:
# Algebraic promotion
import numpy as np
import stim
from tqdm import tqdm
import random
import pickle
defaultzschedule = np.asarray([
    [1, 2, 3, 4, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 2, 3, 4],
    [2, 3, 0, 0, 4, 1, 0, 0],
    [3, 0, 4, 0, 2, 0, 1, 0],
])
defaultxschedule = np.asarray([
    [1, 2, 3, 4, 5, 6, 7, 8]
])
def printnice(shots):
    for i in range(len(shots)):
        print(np.array(shots[i]))

def base4(n):
    digits = [0] * 8
    i = 0
    while n:
        digits[i] = n % 4
        n //= 4
        i += 1
    return digits[::-1]
def base2(n):
    digits = [0] * 8
    i = 0
    while n:
        digits[i] = n % 2
        n //= 2
        i += 1
    return digits[::-1]
def toDecimal(binary):
    return int(binary, 2)
def toBinary(decimal):
    return format(decimal, '08b')
def parity(x):
    p = 0
    while x:
        p ^= 1
        x &= x - 1    # removes the lowest set bit
    return p          # 0 = even, 1 = odd
def parityList(list):
    p = 0
    list = np.array(list, dtype = np.bool)
    for i in list:
        p ^= i
    return p % 2
def measure_logical(log, readout):
    count = 0
    for i in range(len(readout)):
        if readout[i] == 1 and log[i] == '1':
            count += 1
    return count % 2
def logical_to_string(log):
    out = ''
    for i in range(len(log)):
        out += 'Z' if log[i] == '1' else 'I'
    return out
def weight(bitstring):
    return sum([int(bit) for bit in bitstring])
def makeschedule(logicaloperators): # Right now implemented for x stabs only
    schedule = []
    leftoverscheduleentry = []
    leftoverschedulenwindow = 0
    logopindex = 0
    for logop in logicaloperators:
        nwindow = 0
        scheduleentry = []
        if logopindex == 0:
            for i in range(len(logop)):
                if logop[i] == '1':
                    nwindow += 1
                    scheduleentry.append(nwindow)
                    leftoverscheduleentry.append(0)
                else:
                    leftoverschedulenwindow += 1
                    scheduleentry.append(0)
                    leftoverscheduleentry.append(leftoverschedulenwindow)
            logopindex += 1
        else:
            for i in range(len(logop)):
                if logop[i] == '1':
                    nwindow += 1
                    scheduleentry.append(nwindow)
                else:
                    scheduleentry.append(0)
        schedule.append(scheduleentry)
    schedule.append(leftoverscheduleentry)
    #print(schedule)
    return np.array(schedule)


In [2]:
makeschedule(['00001111'])

array([[0, 0, 0, 0, 1, 2, 3, 4],
       [1, 2, 3, 4, 0, 0, 0, 0]])

In [7]:
# Basic simulator
errortypes = ['I', 'X', 'Y', 'Z']
def measure_schedules(circuit, zschedule, xschedule):
    for row in zschedule:
        stabilizerstring = ''
        for i in range(len(row)):
            if row[i] != 0:
                stabilizerstring += 'Z' + str(i) + '*'
        circuit.append('MPP', stim.PauliString(stabilizerstring[:-1]))
    for row in xschedule:
        stabilizerstring = ''
        for i in range(len(row)):
            if row[i] != 0:
                stabilizerstring += 'X' + str(i) + '*'
        circuit.append('MPP', stim.PauliString(stabilizerstring[:-1]))
    return circuit

def simulateError(zschedule = defaultzschedule, xschedule = defaultxschedule, errors = ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'], nshots = 100000):
    circuit = stim.Circuit()
    circuit.append('RX', [0, 1, 2, 3, 4, 5, 6, 7])
    measure_schedules(circuit, zschedule, xschedule)
    for i in range(len(errortypes)):
        qubitlist = []
        for j in range(len(errors)):
            if errors[j] == errortypes[i]:
                qubitlist.append(j)
        circuit.append(errortypes[i], qubitlist)
    measure_schedules(circuit, zschedule, xschedule)
    circuit.append('MX', [0, 1, 2, 3, 4, 5, 6, 7])
    #eturn circuit
    sampler = circuit.compile_sampler()
    shots = sampler.sample(nshots)
    return shots.astype(int), circuit

results, c = simulateError(errors = ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'], nshots = 4)
detector, readout = (results[:, :5] + results[:, 5:10]) % 2, results[:, 10:]
for i in range(len(detector)):
    print(detector[i], readout[i])

def defineSearchSpace(p = 0.1, pthresh = 0.00001):
    errorspace = []
    totalp = (1-p)**8
    coveredp = (1-p)**8
    for i in range(1, 4**8):
        errors = [errortypes[j] for j in base4(i)]
        perror = 1
        for error in errors:
            if error == 'I':
                perror *= 1 - p
            else:
                perror *= (p/3)
        totalp += perror
        if perror >= pthresh:
            errorspace.append(errors)
            coveredp += perror
    print(f'Kept error sequences: {len(errorspace)} out of {4**8}, Covered probability: {coveredp} / {totalp}')
    return errorspace

def sampleErrorSpace(errorspace, zschedule, xschedule, logicaloperators, p = 0.1,nshots = 1, uniformp = True):
    zslen, xslen = zschedule.shape[0], xschedule.shape[0]
    print('zslen', zslen, 'xslen', xslen)
    logopsFailures = {}
    fallthrougherrors = []
    if uniformp:
        stabdegree = np.ones(8)
    else:
        stabdegree = np.zeros(8)
        for stab in zschedule:
            for i in range(8):
                if stab[i] != 0:
                    stabdegree[i] += 1
        for stab in xschedule:
            for i in range(8):
                if stab[i] != 0:
                    stabdegree[i] += 1
    print(stabdegree)
    for errors in errorspace:
        perror = 1
        for i in range(8):
            if errors[i] == 'I':
                perror *= 1 - (stabdegree[i] * p)
            else:
                perror *= (stabdegree[i] * p/3)
        result, c = simulateError(zschedule = zschedule, xschedule = xschedule, errors = errors, nshots = nshots)

        detector, readout = (result[:, :zslen + xslen] + result[:, zslen+xslen:2*(zslen+xslen)]) % 2, result[:, 2*(zslen+xslen):]
        for log in logicaloperators:
            if measure_logical(log, readout[0]) == 1 and np.sum(detector[0]) == 0:
                fallthrougherrors.append(errors)
                if log not in logopsFailures:
                    logopsFailures[log] = perror
                else:
                    logopsFailures[log] += perror
    return logopsFailures, fallthrougherrors

[0 0 0 0 0] [0 0 0 0 1 1 1 1]
[0 0 0 0 0] [1 0 1 0 1 0 1 0]
[0 0 0 0 0] [0 1 1 0 1 0 0 1]
[0 0 0 0 0] [0 1 0 1 1 0 1 0]


In [8]:
# Create set of all logical operators
row, col = 0, 4 

logicaloperators = {}
for i in range(1, 256):
    if parity(i) == 0:
        logicaloperators[toBinary(i)] = 0
print(len(logicaloperators), logicaloperators.keys())


127 dict_keys(['00000011', '00000101', '00000110', '00001001', '00001010', '00001100', '00001111', '00010001', '00010010', '00010100', '00010111', '00011000', '00011011', '00011101', '00011110', '00100001', '00100010', '00100100', '00100111', '00101000', '00101011', '00101101', '00101110', '00110000', '00110011', '00110101', '00110110', '00111001', '00111010', '00111100', '00111111', '01000001', '01000010', '01000100', '01000111', '01001000', '01001011', '01001101', '01001110', '01010000', '01010011', '01010101', '01010110', '01011001', '01011010', '01011100', '01011111', '01100000', '01100011', '01100101', '01100110', '01101001', '01101010', '01101100', '01101111', '01110001', '01110010', '01110100', '01110111', '01111000', '01111011', '01111101', '01111110', '10000001', '10000010', '10000100', '10000111', '10001000', '10001011', '10001101', '10001110', '10010000', '10010011', '10010101', '10010110', '10011001', '10011010', '10011100', '10011111', '10100000', '10100011', '10100101', '

In [8]:
zschedule = defaultzschedule
#zschedule = np.load('schedules/schedule_11000000.npy')
zslen = zschedule.shape[0]
print(zslen, zschedule)

4 [[1 2 3 4 0 0 0 0]
 [0 0 0 0 1 2 3 4]
 [2 3 0 0 4 1 0 0]
 [3 0 4 0 2 0 1 0]]


In [19]:
p = 0.1

logicaloperators = {}
for i in range(1, 256):
    if parity(i) == 0:
        logicaloperators[toBinary(i)] = 0
print(len(logicaloperators), logicaloperators.keys())

ler = 0
conditionaller = 0
it = 5
totalp = (1-p)**8
sneakyreadouts = []
sampledreadouts = 0
for i in tqdm(range(1, 4**8)):
    errors = [errortypes[j] for j in base4(i)]
    perror = 1
    for error in errors:
        if error == 'I':
            perror *= 1 - p
        else:
            perror *= (p/3)
    totalp += perror
    if perror < 0.00001: # skip very rare effects
        continue
    sampledreadouts += 1
    result = simulateError(zschedule = zschedule, errors = errors, nshots = 1)
    #print(result)
    detector, readout = (result[:, :zslen + 1] + result[:, zslen+1:2*zslen+2]) % 2, result[:, 2*zslen+2:]
    #print(detector, readout)
    for log in logicaloperators:
        if measure_logical(log, readout[0]) == 1 and np.sum(detector[0][0:zslen]) == 0:
            logicaloperators[log] += perror
            sneakyreadouts.append(readout[0])
sneakyreadouts = np.array(sneakyreadouts)
sneakyreadouts = np.unique(sneakyreadouts, axis=0)
print(len(sneakyreadouts))
print(sampledreadouts)
print(totalp)

127 dict_keys(['00000011', '00000101', '00000110', '00001001', '00001010', '00001100', '00001111', '00010001', '00010010', '00010100', '00010111', '00011000', '00011011', '00011101', '00011110', '00100001', '00100010', '00100100', '00100111', '00101000', '00101011', '00101101', '00101110', '00110000', '00110011', '00110101', '00110110', '00111001', '00111010', '00111100', '00111111', '01000001', '01000010', '01000100', '01000111', '01001000', '01001011', '01001101', '01001110', '01010000', '01010011', '01010101', '01010110', '01011001', '01011010', '01011100', '01011111', '01100000', '01100011', '01100101', '01100110', '01101001', '01101010', '01101100', '01101111', '01110001', '01110010', '01110100', '01110111', '01111000', '01111011', '01111101', '01111110', '10000001', '10000010', '10000100', '10000111', '10001000', '10001011', '10001101', '10001110', '10010000', '10010011', '10010101', '10010110', '10011001', '10011010', '10011100', '10011111', '10100000', '10100011', '10100101', '

100%|██████████| 65535/65535 [00:00<00:00, 74360.13it/s] 

0
1788
1.000000000000325


In [7]:
result = simulateError(errors = ['X', 'I', 'X', 'I', 'X', 'I', 'X', 'Z'], nshots = 1)
detector, readout = (result[:, :5] + result[:, 5:10]) % 2, result[:, 10:]
print(detector, readout)


[[0 0 0 0 1]] [[1 0 1 0 1 0 1 0]]


In [20]:
failureprobs = {}
sumfailure = 0
for log, val in sorted(logicaloperators.items(), key=lambda x: x[1]):
    print(f"{(log)}: {val}")
    if val > 0:
        if val not in failureprobs:
            failureprobs[val] = [log]
        else:
            failureprobs[val].append(log)
        sumfailure += val
print(failureprobs)
print(sumfailure)


00000011: 0
00000101: 0
00000110: 0
00001001: 0
00001010: 0
00001100: 0
00001111: 0
00010001: 0
00010010: 0
00010100: 0
00010111: 0
00011000: 0
00011011: 0
00011101: 0
00011110: 0
00100001: 0
00100010: 0
00100100: 0
00100111: 0
00101000: 0
00101011: 0
00101101: 0
00101110: 0
00110000: 0
00110011: 0
00110101: 0
00110110: 0
00111001: 0
00111010: 0
00111100: 0
00111111: 0
01000001: 0
01000010: 0
01000100: 0
01000111: 0
01001000: 0
01001011: 0
01001101: 0
01001110: 0
01010000: 0
01010011: 0
01010101: 0
01010110: 0
01011001: 0
01011010: 0
01011100: 0
01011111: 0
01100000: 0
01100011: 0
01100101: 0
01100110: 0
01101001: 0
01101010: 0
01101100: 0
01101111: 0
01110001: 0
01110010: 0
01110100: 0
01110111: 0
01111000: 0
01111011: 0
01111101: 0
01111110: 0
10000001: 0
10000010: 0
10000100: 0
10000111: 0
10001000: 0
10001011: 0
10001101: 0
10001110: 0
10010000: 0
10010011: 0
10010101: 0
10010110: 0
10011001: 0
10011010: 0
10011100: 0
10011111: 0
10100000: 0
10100011: 0
10100101: 0
10100110: 0
1010

In [88]:
# Create set of all logical operators for X
row, col = 0, 4
logicaloperators = {}
for i in range(1, 255):
    if parity(i) == 0:
        xstab = np.array(base2(i), dtype = np.bool)
        addstab = True
        for zstab in defaultzschedule:
            if parityList(zstab * xstab) != 0:
                addstab = False
                break
        if addstab:
            logicaloperators[toBinary(i)] = 0
print(len(logicaloperators), logicaloperators.keys())
# Create set of all single promotion pathways and schedules
singlepromotionschedules = []
for logop in logicaloperators:
    singlepromotionschedules.append(makeschedule([logop]))
print(len(singlepromotionschedules), singlepromotionschedules[0])
with open('singlepromotionschedules.pkl', 'wb') as f:
    pickle.dump(singlepromotionschedules, f)
# Create set of all double promotion pathways and schedules
doublepromotionschedules = []
for i in range(len(logicaloperators.keys())):
    for j in range(i+1, len(logicaloperators.keys())):
        doublepromotionschedules.append(makeschedule([list(logicaloperators.keys())[i], list(logicaloperators.keys())[j]]))
print(len(doublepromotionschedules), doublepromotionschedules[0])
with open('doublepromotionschedules.pkl', 'wb') as f:
    pickle.dump(doublepromotionschedules, f)

14 dict_keys(['00001111', '00110011', '00111100', '01010101', '01011010', '01100110', '01101001', '10010110', '10011001', '10100101', '10101010', '11000011', '11001100', '11110000'])
14 [[0 0 0 0 1 2 3 4]
 [1 2 3 4 0 0 0 0]]
91 [[0 0 0 0 1 2 3 4]
 [0 0 1 2 0 0 3 4]
 [1 2 3 4 0 0 0 0]]


0 1 2


In [10]:
# Check promotion schedules
print(doublepromotionschedules[12])

[[0 0 0 0 1 2 3 4]
 [1 2 3 4 0 0 0 0]
 [1 2 3 4 0 0 0 0]]


In [87]:
# Simulation stack
p = 0.1
ler = 0
errorspace = defineSearchSpace(p = p)
xschedule = singlepromotionschedules[4]
#xschedule = defaultxschedule
#rand = np.random.randint(0, 91)
rand = 6
print(rand)
xschedule = doublepromotionschedules[rand]
print(xschedule)
logopsFailures, fallthrougherrors = sampleErrorSpace(errorspace, defaultzschedule, xschedule, logicaloperators, p = p, nshots = 1, uniformp = False)
for key, value in logopsFailures.items():
    print(f"{key}: {value}")


Kept error sequences: 1788 out of 65536, Covered probability: 0.9949756499999682 / 1.000000000000325
6
[[0 0 0 0 1 2 3 4]
 [1 0 0 2 0 3 4 0]
 [1 2 3 4 0 0 0 0]]
zslen 4 xslen 3
[5. 3. 3. 3. 4. 4. 4. 2.]
00110011: 0.004028266666666666
00111100: 0.004028266666666666
01010101: 0.004028266666666666
01011010: 0.004028266666666666
10100101: 0.004028266666666666
10101010: 0.004028266666666666
11000011: 0.004028266666666666
11001100: 0.004028266666666666


In [265]:
# Check error space
print(len(errorspace))
print(random.choice(errorspace))

1788
['I', 'I', 'Y', 'X', 'I', 'I', 'Y', 'I']


In [282]:
# Check fallthrough errors
print(len(fallthrougherrors))
for i in range(len(fallthrougherrors)):
    out = ''
    for pauli in fallthrougherrors[i]:
        if pauli == 'I':
            out += '0'
        else:
            out += '1'
    print(out)



96
00000101
00000101
00000101
00000101
00000101
00000101
00000101
00000101
00001010
00001010
00001010
00001010
00001010
00001010
00001010
00001010
00010010
00010010
00010010
00010010
00010010
00010010
00010010
00010010
00011000
00011000
00011000
00011000
00011000
00011000
00011000
00011000
00100001
00100001
00100001
00100001
00100001
00100001
00100001
00100001
00100100
00100100
00100100
00100100
00100100
00100100
00100100
00100100
01000010
01000010
01000010
01000010
01000010
01000010
01000010
01000010
01001000
01001000
01001000
01001000
01001000
01001000
01001000
01001000
01010000
01010000
01010000
01010000
01010000
01010000
01010000
01010000
10000001
10000001
10000001
10000001
10000001
10000001
10000001
10000001
10000100
10000100
10000100
10000100
10000100
10000100
10000100
10000100
10100000
10100000
10100000
10100000
10100000
10100000
10100000
10100000


In [195]:
errors = ['Z', 'I', 'I', 'I', 'Z', 'I', 'I', 'I']
#xschedule = singlepromotionschedules[0]
xschedule = defaultxschedule
print(xschedule)
result, c = simulateError(errors = errors, zschedule = defaultzschedule, xschedule = xschedule, nshots = 1)
zslen, xslen = defaultzschedule.shape[0], defaultxschedule.shape[0]
detector, readout = (result[:, :zslen + xslen] + result[:, zslen+xslen:2*(zslen+xslen)]) % 2, result[:, 2*(zslen+xslen):]
print(detector, readout)
print(c.diagram())

[[1 2 3 4 5 6 7 8]]
[[0 0 0 0 0]] [[0 0 1 0 1 1 0 1]]
q0: -RX-MPP[Z]:rec[0]-MPP[Z]:rec[2]-MPP[Z]:rec[3]-MPP[X]:rec[4]-Z-MPP[Z]:rec[5]-MPP[Z]:rec[7]-MPP[Z]:rec[8]-MPP[X]:rec[9]-MX:rec[10]-
        |             |             |             |               |             |             |             |
q1: -RX-MPP[Z]:rec[0]-MPP[Z]:rec[2]-|-------------MPP[X]:rec[4]-I-MPP[Z]:rec[5]-MPP[Z]:rec[7]-|-------------MPP[X]:rec[9]-MX:rec[11]-
        |             |             |             |               |             |             |             |
q2: -RX-MPP[Z]:rec[0]-|-------------MPP[Z]:rec[3]-MPP[X]:rec[4]-I-MPP[Z]:rec[5]-|-------------MPP[Z]:rec[8]-MPP[X]:rec[9]-MX:rec[12]-
        |             |             |             |               |             |             |             |
q3: -RX-MPP[Z]:rec[0]-|-------------|-------------MPP[X]:rec[4]-I-MPP[Z]:rec[5]-|-------------|-------------MPP[X]:rec[9]-MX:rec[13]-
                      |             |             |                             